# Hamiltonian Neural Networks for the 3D Three-Body Problem

This notebook trains two neural networks to predict the motion of three gravitating bodies in 3D, and compares them with each other and with standard numerical integrators.

- A **baseline network** learns the time derivatives of the state directly.
- A **Hamiltonian neural network (HNN)** learns a single scalar function $H$ and gets the time derivatives from Hamilton's equations.

The code extends the 2D experiments of Greydanus, Dzamba and Yosinski, *Hamiltonian Neural Networks* (2019), whose repository this project is forked from.

**Contents**

0. Setup
1. Background: what an HNN is
2. The data: near-circular 3D three-body orbits
3. Training one baseline and one HNN
4. Looking at one pair of models
5. Measured results over 5 random seeds
6. Conclusions and limitations
7. Appendix: the original 2D version

**Main results** (`OUT_OF_PLANE_NOISE = 0.5`, mean ± spread over 5 seeds):

| Measure | Baseline | HNN |
|---|---|---|
| Derivative error on unseen orbits (mean) | 0.0523 ± 0.0002 | 0.0170 ± 0.0010 |
| Relative energy drift at $t = 5$ (median) | 1.30 ± 0.16 | 0.133 ± 0.037 |
| Position error at $t = 5$ (median) | 2.08 ± 0.08 | 0.72 ± 0.16 |

Section 5 explains each measure. The numbers above were typed in from that run; they will differ slightly if you regenerate the data.

If you encounter any issues or bugs not mentioned throughout the script, restarting the kernel and re-running each cell up to that point (excluding dataset-generation or model-training if they were saved, unless you have time to run them too) usually works.

# 0. Setup

One-time setup, from a terminal in the repository root:

```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

Then select the `.venv` interpreter as this notebook's kernel. Run the cells in order. After editing any `.py` file, restart the kernel, because imported modules are kept in memory.

In [ ]:
# Move to the repository root (the folder that holds hnn3d.py), wherever the notebook was opened from
import os, sys
while not os.path.exists('hnn3d.py') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
assert os.path.exists('hnn3d.py'), 'Could not find the repository root'
print('Python executable:', sys.executable)
print('Working directory:', os.getcwd())

In [ ]:
# Load the plotting functions used in sections 2 to 4
import torch   # imported before matplotlib's Qt backend, which otherwise clashes with it on Windows

%matplotlib inline
# For 3D plots you can rotate, use this line instead (plots open in separate windows; needs PyQt5):
# %matplotlib qt

%run experiment-3body/3D/visuals3dcanonical.py

# 1. Background: what an HNN is

**The system.** Three bodies of unit mass attract each other by gravity ($G = 1$). The state is the position $\mathbf{q}_i$ and momentum $\mathbf{p}_i$ of each body, 18 numbers in total, ordered body by body:

$$\mathbf{x} = (\mathbf{q}_1, \mathbf{p}_1, \mathbf{q}_2, \mathbf{p}_2, \mathbf{q}_3, \mathbf{p}_3)$$

**Hamilton's equations.** The total energy written as a function of positions and momenta is the Hamiltonian,

$$H = \sum_{i=1}^{3} \frac{|\mathbf{p}_i|^2}{2m_i} \;-\; \sum_{i<j} \frac{G\, m_i m_j}{|\mathbf{q}_i - \mathbf{q}_j|}$$

and it generates the motion:

$$\frac{d\mathbf{q}_i}{dt} = \frac{\partial H}{\partial \mathbf{p}_i}, \qquad \frac{d\mathbf{p}_i}{dt} = -\frac{\partial H}{\partial \mathbf{q}_i}$$

Any motion generated this way conserves $H$ exactly.

**The two models.** Both are small networks (18 inputs, two hidden layers of 200 units, tanh) trained on pairs of a state $\mathbf{x}$ and its true time derivative $\dot{\mathbf{x}}$.

- The **baseline** outputs 18 numbers and is trained to match $\dot{\mathbf{x}}$ directly. Nothing forces its predictions to conserve energy.
- The **HNN** outputs one number, $H_\theta(\mathbf{x})$. Its prediction is the gradient of that output, rearranged by a fixed matrix $M$ that pairs each position with its own momentum:

$$\hat{\dot{\mathbf{x}}} = \nabla_{\mathbf{x}} H_\theta \; M$$

Whatever $H_\theta$ the network learns, the motion it predicts conserves $H_\theta$. If $H_\theta$ is close to the true energy, the true energy is nearly conserved too.

Both are trained with the same loss, the mean squared error between predicted and true derivatives.

**A note on sign.** With the matrix $M$ as written in `hnn3d.py`, the code computes $\dot{\mathbf{q}} = -\partial H_\theta / \partial \mathbf{p}$ and $\dot{\mathbf{p}} = +\partial H_\theta / \partial \mathbf{q}$, the opposite sign to the equations above. The network compensates by learning approximately $-H$. The predicted motion is unaffected, but $H_\theta$ should be read as minus the energy.

# 2. The data

Each training orbit is generated by `random_config()` in `experiment-3body/3D/data3dcanonical.py`:

1. Place the three bodies on an equilateral triangle of radius between 0.9 and 1.2, with the velocities that would give a perfectly circular orbit.
2. Perturb each velocity by up to 20%.
3. Add a random velocity kick perpendicular to the triangle's plane (size set by `OUT_OF_PLANE_NOISE`), then remove the net momentum so the centre of mass stays still. Without this kick, three bodies never leave their starting plane.
4. Rotate the whole system by a random 3D rotation.

The orbit is then integrated from $t = 0$ to $t = 5$ with SciPy's adaptive Runge-Kutta solver and sampled at 20 times. Orbits in which any derivative component exceeds 20 are discarded and regenerated: these are close encounters between two bodies, where the forces become very large.

The dataset is 5,000 orbits, which gives 100,000 samples: 80,000 for training and 20,000 for testing.

The next cell builds the dataset and saves it as `3body-3Dorbits-dataset.pkl`. Skip it if the file already exists and the settings have not changed.

In [ ]:
%run experiment-3body/3D/make_3d_dataset.py

If the cell above shows no progress or hangs, run the same script in a terminal instead:

```powershell
python experiment-3body/3D/make_3d_dataset.py
```

In [ ]:
# Summary of the saved dataset
import pickle, numpy as np
with open('experiment-3body/3D/3body-3Dorbits-dataset.pkl', 'rb') as f:
    data = pickle.load(f)

print('training samples:', data['coords'].shape, '| test samples:', data['test_coords'].shape)
dc = np.abs(np.concatenate([data['test_dcoords'], data['dcoords']]))
print('largest derivative component:', dc.max().round(2), '(the filter rejects anything above 20)')
print('derivative percentiles (50, 99, 100):', np.percentile(dc, [50, 99, 100]).round(2))
energy = np.concatenate([data['test_energy'], data['energy']])
print('energy range:', energy.min().round(2), 'to', energy.max().round(2))

## One ground-truth orbit

The left panel shows the three trajectories, and the right panel shows the kinetic, potential and total energy. The total energy should be a flat line.

The orbit lies close to a tilted plane. The out-of-plane motion is real but small compared with the size of the orbit, so the cell after the plot measures it directly.

In [ ]:
plot_ground_truth(plot3d=True)

In [ ]:
# How far does that orbit move out of its starting plane?
import numpy as np, data3dcanonical as D
np.random.seed(0)
orbit, _ = D.get_orbit(D.random_config(), t_points=1000, t_span=[0, 20], rtol=1e-9)
q = orbit[:, 1:4, :]                                   # positions, shape (3 bodies, 3, time)
normal = np.cross(q[1, :, 0] - q[0, :, 0], q[2, :, 0] - q[0, :, 0])
normal /= np.linalg.norm(normal)
print('OUT_OF_PLANE_NOISE =', D.OUT_OF_PLANE_NOISE)
print('largest distance out of the starting plane:', np.abs((q * normal[None, :, None]).sum(axis=1)).max().round(2))

# 3. Training one baseline and one HNN

The training script trains the baseline and then the HNN, each for 10,000 steps with batches of 600 samples (Adam optimiser, learning rate $10^{-3}$). It prints the loss every 200 steps and saves both models next to the dataset. This takes a few minutes.

In [ ]:
%run experiment-3body/3D/train3DtestcodeTesting.py --verbose

In [ ]:
# The training script defines its own get_args(), which replaces the one the plotting
# functions use. Reloading the plotting script puts the right one back.
%run experiment-3body/3D/visuals3dcanonical.py

# Load the two trained models
def load_model(args, baseline=False):
    output_dim = args.input_dim if baseline else 1
    nn_model = MLP(args.input_dim, args.hidden_dim, output_dim, args.nonlinearity)
    model = HNN(args.input_dim, differentiable_model=nn_model,
                field_type=args.field_type, baseline=baseline)
    case = 'baseline' if baseline else 'hnn'
    path = "{}/{}-orbits-{}.tar".format(args.save_dir, args.name, case)
    print(path)
    model.load_state_dict(torch.load(path))
    return model

args = ObjectView(get_args())
base_model = load_model(args, baseline=True)
hnn_model = load_model(args, baseline=False)

## Training curves

Both losses should fall and then level off, with the HNN ending lower. Occasional spikes come from batches that happen to contain orbits with large forces.

In [ ]:
plot_training_curves()

# 4. Looking at one pair of models

These plots use the two models trained above. Each model is rolled out from the same starting state as a ground-truth orbit, so the plots show what the models do, for one orbit at a time. They illustrate the behaviour; the measurements are in section 5.

Run the next two cells first. The later plots in this section reuse the orbits they compute.

## Baseline rollout

In [ ]:
what_has_baseline_learned(base_model, args)

## HNN rollout

In [ ]:
what_has_hnn_learned(hnn_model, args)

## Trajectories side by side

Ground truth, baseline and HNN from the same starting state.

In [ ]:
visualize_all_orbits(base_model, hnn_model, args)

## Energies side by side

The grey line in the two model panels is the true, conserved total energy. The further a model's total energy moves from it, the worse the model conserves energy.

In [ ]:
visualize_all_energies(base_model, hnn_model, args)

## Energy conservation over several orbits

Total energy over time for 15 different starting states.

In [ ]:
visualize_energy_conservation(base_model, hnn_model, args)

# 5. Measured results over 5 random seeds

Training a network involves random choices: the starting weights and the order of the batches. One trained model could be better or worse by luck. To separate luck from a real difference, this section trains 5 baselines and 5 HNNs from scratch (seeds 0 to 4) on the same dataset and reports the mean and spread.

The models are saved in `experiment-3body/3D/seed_runs/`, separately from the pair used in sections 3 and 4.

## 5.1 Derivative error on unseen orbits

Each model predicts the time derivatives on a fixed test set of 2,000 new orbits (40,000 samples). The error for one sample is

$$\mathrm{err} = \frac{1}{18}\sum_{i=1}^{18}\left(\hat{\dot{x}}_i - \dot{x}_i\right)^2$$

The test set is generated without the close-encounter filter. Samples with a derivative component above 20 are scored separately as "extreme"; the models never saw anything like them in training.

The next cell trains all 10 models and takes 15 to 20 minutes. Skip it if `seed_runs/` already holds models trained on the current dataset.

In [ ]:
%run experiment-3body/3D/run_seeds.py --test_trials 2000

**Result:**

OUT_OF_PLANE_NOISE = 0.5

| Derivative error | Baseline | HNN | Baseline / HNN, per seed |
|---|---|---|---|
| Median | 0.00558 ± 0.00039 | 0.00244 ± 0.00034 | 1.68 to 2.74 |
| In-range mean | 0.0523 ± 0.00024 | 0.0170 ± 0.00098 | 2.86 to 3.36 |
| Extreme mean | 447 ± 1.3 | 416 ± 4 | 1.06 to 1.09 |

On orbits like the training data, the HNN's mean error is about 3 times lower, on every seed. On close encounters both models fail: their errors are about four orders of magnitude larger.

## 5.2 Energy drift over a rollout

Each model is rolled out from 500 test starting states to $t = 5$ (fixed-step RK4, step 0.01), and the true energy is tracked along the way:

$$\delta(t) = \frac{\lvert E(t) - E(0)\rvert}{\lvert E(0)\rvert}$$

The exact motion has $\delta = 0$, so any drift is error introduced by the model.

In [ ]:
from IPython.display import Image, display
!{sys.executable} experiment-3body/3D/energy_drift.py
display(Image('experiment-3body/3D/seed_runs/energy_drift.png'))

**Result (17 November 2026 run):** the median drift at $t = 5$ is 0.133 ± 0.037 for the HNN and 1.30 ± 0.16 for the baseline. The HNN is 7 to 12 times better on every seed. A drift of 13% is still large in absolute terms.

## 5.3 Comparison with numerical methods

The same 500 orbits are rolled out with four methods, all at step 0.01:

- the **baseline** and the **HNN**, which learned the dynamics from data
- **RK4** and **leapfrog**, standard integrators that are given the true equations of motion

Each is scored on energy drift, on position error (the RMS distance of the three bodies from a high-accuracy reference solution), and on run time.

In [ ]:
from IPython.display import Image, display
!{sys.executable} experiment-3body/3D/compare_methods.py
display(Image('experiment-3body/3D/seed_runs/compare_methods.png'))

**Result:**

OUT_OF_PLANE_NOISE = 0.5

| Method | Energy drift at $t = 5$ | Position error at $t = 5$ | Seconds per 500 rollouts |
|---|---|---|---|
| Baseline NN | 1.30 ± 0.16 | 2.08 ± 0.08 | 0.94 |
| HNN | 0.133 ± 0.037 | 0.72 ± 0.16 | 1.71 |
| Leapfrog, true equations | $6.7\times10^{-6}$ | $8.5\times10^{-5}$ | 0.26 |
| RK4, true equations | $8.3\times10^{-8}$ | $3.6\times10^{-10}$ | 0.70 |

The integrators that know the true equations are four to nine orders of magnitude more accurate than either network, and no slower. The useful comparison is between the two learned models: the HNN is about 10 times better on energy drift and about 3 times better on position error.

RK4's energy drift sits at a measurement floor of about $8\times10^{-8}$. The equations use slightly softened gravity, so the plain energy formula is conserved only to that level.

# 6. Conclusions and limitations

**Conclusions**

- When the dynamics have to be learned from data, building in the Hamiltonian structure helps. The HNN predicts derivatives about 3 times more accurately than a plain network of the same size, and its rollouts drift about 10 times less in energy. Both results hold on all 5 seeds.
- A learned model does not compete with a numerical integrator that is given the true equations. The comparison in 5.3 measures the cost of learning the dynamics, and the HNN pays much less of it than the baseline.

**Limitations**

- **Close encounters.** Orbits with very large forces were removed from training, and neither model generalises to them.
- **Absolute accuracy.** By $t = 5$ the HNN's energy is off by about 13% and its positions by about 0.7, on an orbit of radius about 1.
- **Time span.** Rollouts stop at $t = 5$, the end of the training orbits. Nothing here tests longer times.
- **Seeds vary the training only.** All seeds share one training set and one test set, so the spread does not include variation in the data.
- **Degree of 3D motion.** The results depend on `OUT_OF_PLANE_NOISE`. At 0.2 the HNN's lead was larger (3.5 times on derivative error, about 20 times on energy drift).

# 7. Appendix: the original 2D version

These cells run the 2D three-body experiment that the 3D version was extended from. They are kept for reference.

Running them replaces the plotting functions, `args`, `base_model` and `hnn_model` loaded above with their 2D versions. To return to the 3D sections afterwards, restart the kernel and run from section 0.

In [ ]:
# Creates dataset for model training and visualization
!{sys.executable} experiment-3body/2D/make_dataset.py

In [ ]:
# Prints dataset and gives info regarding each feature
!{sys.executable} experiment-3body/2D/show_dataset.py

In [ ]:
!{sys.executable} experiment-3body/2D/train.py --verbose

In [ ]:
%run experiment-3body/2D/visuals.py

In [ ]:
# Loads baseline and hnn models (models need to be trained first before visualizations related to models can be displayed)

def load_model(args, baseline=False):
    output_dim = args.input_dim if baseline else 2
    nn_model = MLP(args.input_dim, args.hidden_dim, output_dim, args.nonlinearity)
    model = HNN(args.input_dim, differentiable_model=nn_model,
            field_type=args.field_type, baseline=baseline)
    case = 'baseline' if baseline else 'hnn'
    path = "{}/{}-orbits-{}.tar".format(args.save_dir, args.name, case)
    print(path)
    model.load_state_dict(torch.load(path))
    return model

args = ObjectView(get_args())
base_model = load_model(args, baseline=True)
hnn_model = load_model(args, baseline=False)

In [ ]:
plot_ground_truth()

In [ ]:
what_has_baseline_learned(base_model, args)

In [ ]:
what_has_hnn_learned(hnn_model, args)